# Neural Sequence Model: Char-CNN BiLSTM-CRF



In this notebook, I implement a more sophisticated architecture for the NER task, specifically following the Ma and Hovy (2016) approach. I decided to move away from a basic BiLSTM + Softmax setup because NER requires a better way to handle rare words and label consistency.

- Character embeddings for each word
- CNN over characters to create a character-level word representation
- Pretrained GloVe word embeddings
- Hybrid Embeddings concatenating those CNN-generated character representations with pretrained GloVe word embeddings.
- BiLSTM to read left and right context
- CRF output layer to decode the best label sequence

This architecture is useful for NER because named entities often contain rare or unseen words(OOV words). The character CNN handles the variety, while the CRF ensures the final output is a sequence that actually makes sense.

## 1. Setup

In [ ]:
import sys
import os
from collections import Counter
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchcrf import CRF

from tqdm import tqdm
import copy

from itertools import product


from seqeval.metrics import f1_score as seq_f1_score
from seqeval.metrics import precision_score as seq_precision_score
from seqeval.metrics import recall_score as seq_recall_score
from seqeval.metrics.sequence_labeling import get_entities
from seqeval.scheme import IOB2

In [110]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

In [111]:
set_seed(42)

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

sys.path.append(os.path.abspath(".."))

## 2. Load Data

In [2]:
from src.load_dataset import load_dataset

train_sents, train_tags = load_dataset("../data/raw/train.txt")
valid_sents, valid_tags = load_dataset("../data/raw/valid.txt")
test_sents, test_tags = load_dataset("../data/raw/test.txt")

print(f"Training sentences: {len(train_sents)}")
print(f"Valid sentences: {len(valid_sents)}")
print(f"Testing sentences: {len(test_sents)}")

Training sentences: 14041
Valid sentences: 3250
Testing sentences: 3453


In [3]:
print(train_sents[:5])
print(valid_tags[:5])
print(test_sents[:5])

[['EU', 'rejects', 'German', 'call', 'to', 'boycott', 'British', 'lamb', '.'], ['Peter', 'Blackburn'], ['BRUSSELS', '1996-08-22'], ['The', 'European', 'Commission', 'said', 'on', 'Thursday', 'it', 'disagreed', 'with', 'German', 'advice', 'to', 'consumers', 'to', 'shun', 'British', 'lamb', 'until', 'scientists', 'determine', 'whether', 'mad', 'cow', 'disease', 'can', 'be', 'transmitted', 'to', 'sheep', '.'], ['Germany', "'s", 'representative', 'to', 'the', 'European', 'Union', "'s", 'veterinary', 'committee', 'Werner', 'Zwingmann', 'said', 'on', 'Wednesday', 'consumers', 'should', 'buy', 'sheepmeat', 'from', 'countries', 'other', 'than', 'Britain', 'until', 'the', 'scientific', 'advice', 'was', 'clearer', '.']]
[['O', 'O', 'B-ORG', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O'], ['B-LOC', 'O'], ['B-MISC', 'I-MISC', 'O', 'B-PER', 'I-PER', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-ORG', 'O', 'B-ORG', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O'],

## 3. Build Word, Character, and Label Vocabularies

The model needs three vocabularies:

- a word vocabulary for word embeddings
- a character vocabulary for the character CNN
- a tag vocabulary for the NER labels

I lowercase words for the word vocabulary to improve coverage with pretrained embeddings. However, I still preserve the original token form when building character inputs. This is important because the character CNN is meant to capture subword and morphological signals such as capitalisation, suffixes, punctuation, and word shape.


In [4]:
# To count each word appears in the training set
word_counter = Counter(word.lower() for sent in train_sents for word in sent)

# To count each character appears in the training set
char_counter = Counter(ch for sent in train_sents for word in sent for ch in word)

# To extract sorted tag list from the training set
tag_list = sorted(set(tag for sent_tags in train_tags for tag in sent_tags))

I sorted the tags to keep the ID mapping consistent. Since sets are unordered, sorting ensures the same tag gets the same index every time I run the code.

In [5]:
PAD_TOKEN = '<PAD>'
UNK_TOKEN = '<UNK>'
PAD_CHAR = '<PAD>'
UNK_CHAR = '<UNK>'


# Initialize dictionaries with PAD_TOKEN and UNK_TOKEN
word_to_id = {PAD_TOKEN: 0, UNK_TOKEN: 1}
char_to_id = {PAD_CHAR: 0, UNK_CHAR: 1}

# Append each word to word_to_id and give ID
for word, count in word_counter.items():
    word_to_id[word] = len(word_to_id)

# Append each word to char_to_id and give ID
for ch, count in char_counter.items():
    char_to_id[ch] = len(char_to_id)

# Assign an ID for each tag
tag_to_id = {tag: idx for idx, tag in enumerate(tag_list)}


In [6]:
print(f'Word vocab size: {len(word_to_id)}')
print(f'Char vocab size: {len(char_to_id)}')
print(f'Number of tags: {len(tag_to_id)}')
print(tag_to_id)

Word vocab size: 21011
Char vocab size: 86
Number of tags: 9
{'B-LOC': 0, 'B-MISC': 1, 'B-ORG': 2, 'B-PER': 3, 'I-LOC': 4, 'I-MISC': 5, 'I-ORG': 6, 'I-PER': 7, 'O': 8}


In [24]:
id_to_tag = {idx: tag for idx, tag in enumerate(tag_list)}

In [25]:
id_to_tag

{0: 'B-LOC',
 1: 'B-MISC',
 2: 'B-ORG',
 3: 'B-PER',
 4: 'I-LOC',
 5: 'I-MISC',
 6: 'I-ORG',
 7: 'I-PER',
 8: 'O'}

I convert words,characters and tags into IDs because the model cannot work with text, only numbers. 

I set "0" **<PAD>** to handle padding when sentences have different lengths, and set it to 0 so it can be ignored during training. 

Secondly, add **<UNK>** to handle unseen words or characters, and set it to 1 , so the model can still learn a representation for unknown inputs. 

Lastly, I assign a unique ID to each word, character and tags from the training data so everything converted into numerical form for the model.

## 4. Dataset and Padding

This step is all about organizing our raw lists into structured **batches**.

In [7]:
# Converts the word into the index
def word_to_index(word):
    # Assigned default value to 1, 
    # If word not in dictionary treat it UNKNOWN WORD
    return word_to_id.get(word.lower(), word_to_id[UNK_TOKEN])

# Converts the character into the index 
# and create the sentence with char ID's
def chars_to_indices(word):
    return [char_to_id.get(ch, char_to_id[UNK_CHAR]) for ch in word]

# Pytorch wrapper
class NERDataset(Dataset):
    def __init__(self, sentences, tags):
        self.sentences = sentences
        self.tags = tags

    # Define to access length of sentences
    def __len__(self):
        return len(self.sentences)

    # Define to access tags and sentences using indexing
    def __getitem__(self, idx):
        words = self.sentences[idx]
        tags = self.tags[idx]
        
        word_ids = [word_to_index(word) for word in words] # Extract word index
        char_ids = [chars_to_indices(word) for word in words] # Extract char index
        tag_ids = [tag_to_id[tag] for tag in tags] # # Extract tag index

        return {'words': words, # word
                'word_ids': word_ids, # word id, input for word embedding
                'char_ids': char_ids, # char id, input for char embedding
                'tag_ids': tag_ids # tag id
                }

In [8]:
from src.collate_batches import collate_batches
from functools import partial

# Creates collate_fn function assigned tag_to_id parameter
collate_fn = partial(collate_batches, tag_to_id=tag_to_id)

batch_size = 32 # Initialize batch size

# Create train loader, validation loader and test loader
train_loader = DataLoader(NERDataset(train_sents, train_tags), batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
valid_loader = DataLoader(NERDataset(valid_sents, valid_tags), batch_size=batch_size, shuffle=False, collate_fn=collate_fn)
test_loader = DataLoader(NERDataset(test_sents, test_tags), batch_size=batch_size, shuffle=False, collate_fn=collate_fn)

I create the NERDataset class as a PyTorch wrapper. Inside `__getitem__`, I take one sentence and convert its words, characters, and tags into their corresponding IDs and it return everything in a dictionary.

After that, implement a custom collate_batches function to handle batching and padding. I use batch size 32, practical choice that works well in most NLP tasks and it is a good balance between speed and memory. I also use dynamic padding so that each batch is padded only to its maximum length, which makes training more efficient and avoids unnecessary padding. 

The mask ensures that padded positions do not affect the model. This is important because the BiLSTM and CRF layers should only learn from actual tokens, not padded values.

Lastly, I shuffle the training data to help the model generalise better, but I do not shuffle validation and test data so the evaluation stays consistent.

In [9]:
example_batch = next(iter(train_loader))
print(f"word_ids: {example_batch['word_ids'].shape}")
print(f"char_ids: {example_batch['char_ids'].shape}")
print(f"tag_ids: {example_batch['tag_ids'].shape}")
print(f"mask: {example_batch['mask'].shape}")

word_ids: torch.Size([32, 47])
char_ids: torch.Size([32, 47, 13])
tag_ids: torch.Size([32, 47])
mask: torch.Size([32, 47])


In [10]:
len(example_batch['words'][0])

10

In [11]:
print(f"sentence: {example_batch['words'][0]}")
print(example_batch['mask'][0])
print(example_batch['word_ids'][0])
print(example_batch['char_ids'][0])

sentence: ['SOCCER', '-', 'RUSSIA', 'AND', 'BRAZIL', 'DRAW', '2-2', 'IN', 'FRIENDLY', '.']
tensor([ True,  True,  True,  True,  True,  True,  True,  True,  True,  True,
        False, False, False, False, False, False, False, False, False, False,
        False, False, False, False, False, False, False, False, False, False,
        False, False, False, False, False, False, False, False, False, False,
        False, False, False, False, False, False, False])
tensor([1756,  637, 1157,   85, 2696, 2661, 2353,  230, 7221,   10,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0])
tensor([[26, 56, 37, 37,  2, 25,  0,  0,  0,  0,  0,  0,  0],
        [31,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
        [25,  3, 26, 26, 61, 65,  0,  0,  0,  0,  0,  0,  0],
        [65, 48, 64,  0,  0,  0,  0,  

In [12]:
example_batch['char_ids'][0].shape

torch.Size([47, 13])

## 5. Pretrained GloVe word embeddings

I use pretrained GloVe embeddings to provide the model with a solid foundation of word meanings learned from a massive external corpus. This is a crucial step for my project because the initial OOV analysis revealed a significant gap. About 12.18% of words in the test set and 8.36% in the validation set were never seen during training. 

When I look at entity types, 

-   the PER category has a very high OOV rate, reaching 54.02% in the test set. This means many person names are not seen during training. 
-   ORG entities also have relatively high OOV rates 23.32%, showing that some categories are more sparse than others.

Because of this, if the model only relies on training data, it may struggle with unseen words. Pretrained embeddings help solve this by placing similar words closer in the embedding space, so the model can generalise better.

When it comes to the "why I choose GloVe instead of Word2Vec",  because it uses global word co-occurrence information, which gives more stable semantic representations. Also, GloVe embeddings are widely available in standard formats and are easy to use.

Furthermore, based on the dataset statistics, 

- The training set contains around 203k tokens across approximately 14k sentences, indicating a small to moderate size dataset. 
- Sentence lengths are relatively short, with a median of around 9–11 tokens and 95% of sentences below 40 tokens.

This analysis guides my choice of embedding dimension.

Using very high-dimensional embeddings, softened the model with unnecessary extra parameters. Since the dataset is not very large, I use 100d embedding as they provide enough capacity to represent word meaning while keeping the model efficient and reducing the risk of overfitting.


In [13]:
with open("../data/glove/wiki_giga_2024_100.txt", encoding="utf-8") as f:
    for i in range(5):
        print(f.readline())

the 0.306717 -0.32053 -0.39364699999999997 0.08282600000000001 0.073522 -0.409154 -0.265564 -0.23693999999999998 -0.305832 0.74529 0.214341 0.27678099999999994 -0.152797 -0.127524 0.11952500000000002 0.640965 -0.175869 0.160711 0.47797799999999996 -0.160939 -0.150093 0.674601 -0.099565 0.021881999999999985 -0.032770999999999995 0.368641 -0.08701900000000007 -0.13332599999999997 0.170143 0.15693399999999996 0.6775059999999999 -0.099686 0.392113 0.37343400000000004 -5.736062 0.413845 0.477368 -0.04169700000000001 0.38310900000000003 0.12015199999999998 -0.20947 0.605104 0.23635299999999998 0.15113100000000002 -0.508865 0.671239 -0.300263 -0.267927 2.549487 0.06717699999999999 0.217224 -0.031316 0.05231 0.119321 -0.332154 -0.8079040000000001 -0.546453 -0.04439199999999999 -0.281657 0.286647 0.32577500000000004 -0.021960000000000007 -0.636903 -0.268063 0.247956 -0.402493 0.276707 -0.275139 0.20115899999999998 0.08284399999999997 0.591695 -0.017126999999999948 -0.09226899999999999 0.3920079

I initialise embeddings for unseen words using a normal distribution with a small standard deviation 0.5, so that they are similar in scale to pretrained GloVe vectors, which are typically in the range of approximately [-1, 1] with a standard deviation around 0.5–0.6. Instead of assigning identical zero vectors to all unknown words, this helps the model learn meaningful representations. 

`float32`-4 bytes per value- is used instead of float64 -8 bytes per value- because it is more efficient in terms of memory and computation. Float64 doubles the memory usage and slows down training. Also, GPUs and Pytorch are optimized for float32.

In [14]:
from src.load_dataset import load_glove

word_embeddings, glove_found = load_glove("../data/glove/wiki_giga_2024_100.txt", word_to_id)

During implementation, I encountered a case where a GloVe vectors (102d) did not match the expected embedding dimension. ValueError occured when constructing the embedding matrix. To handle this, I added a dimension check to ensure only valid vectors are used.

In [15]:
vocab = len(word_to_id)
print(f"Matched GloVe vectors: {glove_found}")
print(f"Coverage: {(glove_found / vocab):.2%}")

Matched GloVe vectors: 18327
Coverage: 87.23%


The pretrained GloVe embeddings cover a substantial portion of the vocabulary, with 18,327 words successfully matched 87.32%. This suggests that most frequent words benefit from pretrained semantic representations.

However, some words remain unmatched and are initialised randomly, which may affect performance, especially for rare entities such as person names.

## 6. Model Architecture

1. Each word is represented with a word embedding.
2. Each word also gets a character representation from a CNN over character embeddings.
3. The word embedding and character representation are concatenated.
4. A BiLSTM reads the sentence from both directions.
5. A CRF layer decodes the best BIO label sequence.

In [16]:
class CharCNN(nn.Module):

    def __init__(self, num_chars, char_embedding_dim, char_out_channels, kernel_size=3, padding_idx=0):
        super().__init__()

        self.char_embedding = nn.Embedding(num_chars, char_embedding_dim, padding_idx=padding_idx)

        self.cnn = nn.Conv1d(in_channels=char_embedding_dim, out_channels=char_out_channels, kernel_size=kernel_size,
                             padding=kernel_size // 2)

        self.relu = nn.ReLU()


    def forward(self, char_ids):
        batch, sent_len, word_len = char_ids.shape

        flatten_chars = char_ids.reshape(batch * sent_len, word_len)
        char_emb = self.char_embedding(flatten_chars)
        char_emb = char_emb.permute(0, 2, 1)
        cnn_out = self.relu(self.cnn(char_emb))

        pooled = torch.max(cnn_out, dim=2).values

        char_rep = pooled.reshape(batch, sent_len, -1)

        return char_rep

In [45]:
class CharCNNBiLSTMCRF(nn.Module):

    def __init__(self,word_embeddings, num_char, num_tags, 
                char_embedding_dim=30, char_out_channel=50,
                lstm_hidden_dim = 200, dropout=0.2):

        super().__init__()

        self.word_emb = nn.Embedding.from_pretrained(word_embeddings, freeze=False, padding_idx=word_to_id[PAD_TOKEN])
        
        self.char_cnn = CharCNN(num_chars=num_char, char_embedding_dim=char_embedding_dim, char_out_channels=char_out_channel,
                                padding_idx=char_to_id[PAD_CHAR])

        self.dropout = nn.Dropout(dropout)

        self.bilstm = nn.LSTM(input_size=word_embeddings.shape[1] + char_out_channel, hidden_size=lstm_hidden_dim,
                              batch_first=True, bidirectional=True)

        # Convert each output into emissions for each tag
        self.emissions = nn.Linear(lstm_hidden_dim * 2, num_tags)

        self.crf = CRF(num_tags, batch_first=True)


    def forward(self, word_ids, char_ids, tags=None, mask=None):
        
        word_emb = self.word_emb(word_ids)  # [B, S, word_emb_dim]
        char_emb = self.char_cnn(char_ids)  # [B, S, char_out_channels]

        x = torch.cat([word_emb, char_emb], dim=-1)

        bilstm_out, _ = self.bilstm(x) # [B, S, hidden_dim]
        bilstm_out = self.dropout(bilstm_out)

        emissions = self.emissions(bilstm_out)

        if tags is not None:

            loss = -self.crf(emissions, tags, mask=mask.bool(), reduction="token_mean")

            return loss

        preds = self.crf.decode(emissions, mask=mask.bool())

        return preds

## 7. Training Loop

In [38]:
def move_to_device(batch, device):

    return {key: value.to(device) if torch.is_tensor(value) else value for key, value in batch.items()}

In [61]:
def train(model, train_data, val_data, lr, epochs):

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model = model.to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    epoch_results = []
    best_val_f1 = -1
    best_model_state = None
    best_epoch = 0
    patience = 3
    epochs_no_improve = 0

    for epoch_num in range(epochs):

        total_loss_train = 0
        train_preds = []
        train_true_labels = []

        model.train()

        for batch in tqdm(train_data):

            batch = move_to_device(batch, device)

            word_ids = batch["word_ids"]
            char_ids = batch["char_ids"]
            labels = batch["tag_ids"]
            mask = batch["mask"].bool()

            optimizer.zero_grad()
            
            
            loss = model(word_ids, char_ids, tags=labels, mask=mask)

            loss.backward()
            optimizer.step()      

            total_loss_train += loss.item()

            with torch.no_grad():

                pred_ids = model(word_ids, char_ids, mask=mask)

                for true_labels, pred_labels, sent_mask in zip(labels.cpu().tolist(), pred_ids, mask.cpu().tolist()):

                    length = sum(sent_mask)

                    train_true_labels.append([id_to_tag[label] for label in true_labels[:length]])
                    train_preds.append([id_to_tag[label] for label in pred_labels[:length]])

        train_f1 = seq_f1_score(train_true_labels, train_preds, mode="strict", scheme=IOB2, average="macro", zero_division=0)


        model.eval()

        total_loss_val = 0
        val_true_labels = []
        val_preds = []

        with torch.no_grad():

            for batch in val_data:
            
                batch = move_to_device(batch, device)
                word_ids = batch["word_ids"]
                char_ids = batch["char_ids"]
                labels = batch["tag_ids"]
                mask = batch["mask"].bool()

                val_loss = model(word_ids, char_ids, tags=labels, mask=mask)

                pred_ids = model(word_ids, char_ids, mask=mask)

                total_loss_val += val_loss.item()

                for true_labels, pred_labels, sent_mask in zip(labels.cpu().tolist(), pred_ids, mask.cpu().tolist()):

                    length = sum(sent_mask)

                    val_true_labels.append([id_to_tag[label] for label in true_labels[:length]])

                    val_preds.append([id_to_tag[label] for label in pred_labels[:length]])

            val_f1 = seq_f1_score(val_true_labels, val_preds, mode="strict", scheme=IOB2, average="macro", zero_division=0)

            # save best model weights by validation macro F1
            if val_f1 > best_val_f1:
                best_val_f1 = val_f1
                best_epoch = epoch_num + 1
                best_model_state = copy.deepcopy(model.state_dict())
                epochs_no_improve = 0

            else:
                epochs_no_improve += 1


            results = {"Epoch": epoch_num + 1, 
                       "train_loss": round(total_loss_train / len(train_data), 3), 
                       "val_loss": round(total_loss_val / len(val_data), 3),
                       "val_f1": round(val_f1, 3),
                       "train_f1": round(train_f1, 3)}


            epoch_results.append(results)

            print(f'Epochs: {epoch_num+1} | Train Loss: {total_loss_train / len(train_data):.3f} | Train F1: {train_f1:.3f}')
            print(f'Val loss: {total_loss_val/len(val_data):.3f} | Val F1: {val_f1:.3f}')

            if epochs_no_improve >= patience:

                print(f"Early stopping after epoch {epoch_num + 1}")
                break

    # Load best weights 
    model.load_state_dict(best_model_state)

    print(f"Best model from epoch {best_epoch} Val Macro F1 = {best_val_f1:.3f}")

    return model, epoch_results, best_epoch

In [57]:
model = CharCNNBiLSTMCRF(word_embeddings=word_embeddings, num_char=len(char_to_id), num_tags=len(tag_to_id), 
                        char_embedding_dim=30, char_out_channel=50,
                        lstm_hidden_dim = 200, dropout=0.2).to(device)


EPOCHS = 10
LR = 1e-3

In [ ]:
model, results = train(model, train_loader, valid_loader, LR, EPOCHS)

100%|██████████| 439/439 [00:27<00:00, 15.82it/s]


Epochs: 1 | Train Loss: 0.250 | Train F1: 0.687
Val loss: 0.100 | Val F1: 0.823


100%|██████████| 439/439 [00:27<00:00, 15.74it/s]


Epochs: 2 | Train Loss: 0.056 | Train F1: 0.907
Val loss: 0.065 | Val F1: 0.863


100%|██████████| 439/439 [00:29<00:00, 14.95it/s]


...

Epochs: 5 | Train Loss: 0.012 | Train F1: 0.982
Val loss: 0.080 | Val F1: 0.872


100%|██████████| 439/439 [00:27<00:00, 16.09it/s]


Epochs: 6 | Train Loss: 0.007 | Train F1: 0.990
Val loss: 0.069 | Val F1: 0.863


100%|██████████| 439/439 [00:27<00:00, 15.79it/s]


Epochs: 7 | Train Loss: 0.005 | Train F1: 0.995
Val loss: 0.073 | Val F1: 0.862
Early stopping after epoch 7
Best model from epoch 4 Val Macro F1 = 0.876


The CharCNN-BiLSTM-CRF model achieved a best validation macro F1 score of `0.876` at epoch 4 and the model learned quickly.

After `epoch 4`, the validation F1 started to decrease while the training F1 continued to improve. This suggests that the model began to overfit the training data. Early stopping helped prevent further overfitting by selecting the best-performing model on the validation set.

The model performs well, but the gap `0.119` between `training F1 0.995` and `validation F1 0.876` indicates some overfitting. Further hyperparameter tuning may improve generalization slightly.

In [59]:
param_lr = [1e-3, 1e-4, 5e-4]
dropouts = [0.2, 0.3, 0.5]

In [62]:
search_results = []

for lr, dropout in product(param_lr, dropouts):

    model = CharCNNBiLSTMCRF(word_embeddings=word_embeddings, num_char=len(char_to_id), num_tags=len(tag_to_id), 
                            char_embedding_dim=30, char_out_channel=50,
                            lstm_hidden_dim = 200, dropout=dropout).to(device)

    _, results, best_epoch = train(model, train_loader, valid_loader, lr, EPOCHS)

    best_epoch_result = results[best_epoch - 1]

    search_results.append({"learning_rate": lr,
                           "dropout": dropout,
                           "Best Epoch": best_epoch,
                           "Train Loss": best_epoch_result["train_loss"],
                           "Val Loss": best_epoch_result["val_loss"],
                           "Train F1": best_epoch_result["train_f1"],
                           "Validation F1": best_epoch_result["val_f1"],
                           "Train-Val Gap": (best_epoch_result["train_f1"] - best_epoch_result["val_f1"])})  


100%|██████████| 439/439 [00:27<00:00, 16.15it/s]


Epochs: 1 | Train Loss: 0.248 | Train F1: 0.690
Val loss: 0.095 | Val F1: 0.840


100%|██████████| 439/439 [00:29<00:00, 14.72it/s]


Epochs: 2 | Train Loss: 0.057 | Train F1: 0.907
Val loss: 0.071 | Val F1: 0.859


100%|██████████| 439/439 [00:26<00:00, 16.36it/s]


...

Epochs: 7 | Train Loss: 0.018 | Train F1: 0.969
Val loss: 0.109 | Val F1: 0.849


100%|██████████| 439/439 [00:28<00:00, 15.52it/s]


Epochs: 8 | Train Loss: 0.013 | Train F1: 0.977
Val loss: 0.111 | Val F1: 0.848


100%|██████████| 439/439 [00:28<00:00, 15.27it/s]


Epochs: 9 | Train Loss: 0.010 | Train F1: 0.983
Val loss: 0.130 | Val F1: 0.851
Early stopping after epoch 9
Best model from epoch 6 Val Macro F1 = 0.855


In [63]:
results_df = pd.DataFrame(search_results).sort_values(by=["Validation F1", "Train-Val Gap"], 
                                                      ascending=[False, True])

results_df

,learning_rate,dropout,Best Epoch,Train Loss,Val Loss,Train F1,Validation F1,Train-Val Gap
0,0.0010,0.2,4,0.019,0.063,0.971,0.878,0.093
1,0.0010,0.3,4,0.020,0.074,0.968,0.872,0.096
7,0.0005,0.3,4,0.038,0.070,0.936,0.869,0.067
6,0.0005,0.2,4,0.038,0.068,0.937,0.869,0.068
2,0.0010,0.5,6,0.010,0.081,0.983,0.869,0.114
8,0.0005,0.5,6,0.024,0.092,0.959,0.855,0.104
4,0.0001,0.3,10,0.058,0.094,0.896,0.829,0.067
5,0.0001,0.5,9,0.066,0.100,0.880,0.823,0.057
3,0.0001,0.2,10,0.059,0.116,0.898,0.822,0.076


I used grid search to test different learning rates and dropout values.

The best result was achieved with `learning rate of 0.001 and a dropout rate of 0.2`, giving `the validation macro F1 score of 0.878`. Lower learning rates did not perform as well, and increasing dropout did not lead to any improvement.

The validation score only increased slightly, but the gap between training and validation F1 decreased from `0.119 to 0.093`. This suggests that the model was generalising a bit better and relying less on memorising the training data.

Based on these results, I use `learning rate of 0.001 and dropout of 0.2` for the further experiments.

### Additional Dropout Regularisation

To reduce the gap, an additional dropout layer was applied before the BiLSTM.

In [64]:
class CharCNNBiLSTMCRF2(nn.Module):

    def __init__(self,word_embeddings, num_char, num_tags, 
                char_embedding_dim=30, char_out_channel=50,
                lstm_hidden_dim = 200, dropout=0.2):

        super().__init__()

        self.word_emb = nn.Embedding.from_pretrained(word_embeddings, freeze=False, padding_idx=word_to_id[PAD_TOKEN])
        
        self.char_cnn = CharCNN(num_chars=num_char, char_embedding_dim=char_embedding_dim, char_out_channels=char_out_channel,
                                padding_idx=char_to_id[PAD_CHAR])

        self.dropout = nn.Dropout(dropout)

        self.bilstm = nn.LSTM(input_size=word_embeddings.shape[1] + char_out_channel, hidden_size=lstm_hidden_dim,
                              batch_first=True, bidirectional=True)

        # Convert each output into emissions for each tag
        self.emissions = nn.Linear(lstm_hidden_dim * 2, num_tags)

        self.crf = CRF(num_tags, batch_first=True)


    def forward(self, word_ids, char_ids, tags=None, mask=None):
        
        word_emb = self.word_emb(word_ids)  # [B, S, word_emb_dim]
        char_emb = self.char_cnn(char_ids)  # [B, S, char_out_channels]

        x = torch.cat([word_emb, char_emb], dim=-1)
        x = self.dropout(x)

        bilstm_out, _ = self.bilstm(x) # [B, S, hidden_dim]
        bilstm_out = self.dropout(bilstm_out)

        emissions = self.emissions(bilstm_out)

        if tags is not None:

            loss = -self.crf(emissions, tags, mask=mask.bool(), reduction="token_mean")

            return loss

        preds = self.crf.decode(emissions, mask=mask.bool())

        return preds

In [66]:
model2 = CharCNNBiLSTMCRF2(word_embeddings=word_embeddings, num_char=len(char_to_id), num_tags=len(tag_to_id), 
                        char_embedding_dim=30, char_out_channel=50,
                        lstm_hidden_dim = 200, dropout=0.2).to(device)


EPOCHS = 10
LR = 1e-3

In [67]:
model2, results, best_epoch = train(model2, train_loader, valid_loader, LR, EPOCHS)

100%|██████████| 439/439 [00:32<00:00, 13.55it/s]


Epochs: 1 | Train Loss: 0.285 | Train F1: 0.635
Val loss: 0.104 | Val F1: 0.820


100%|██████████| 439/439 [00:29<00:00, 14.91it/s]


Epochs: 2 | Train Loss: 0.073 | Train F1: 0.874
Val loss: 0.085 | Val F1: 0.848


100%|██████████| 439/439 [00:28<00:00, 15.23it/s]


...

Epochs: 5 | Train Loss: 0.022 | Train F1: 0.959
Val loss: 0.065 | Val F1: 0.873


100%|██████████| 439/439 [00:28<00:00, 15.54it/s]


Epochs: 6 | Train Loss: 0.016 | Train F1: 0.970
Val loss: 0.068 | Val F1: 0.875


100%|██████████| 439/439 [00:27<00:00, 15.70it/s]


Epochs: 7 | Train Loss: 0.012 | Train F1: 0.977
Val loss: 0.079 | Val F1: 0.861
Early stopping after epoch 7
Best model from epoch 4 Val Macro F1 = 0.880


After adding dropout before the BiLSTM, the model achieved a best validation macro `F1 score of 0.880`, slightly improving on the previous result of `0.878`.

The train-validation gap also decreased from `0.093 to 0.083`, suggesting that the additional dropout helped reduce overfitting. As expected, the training F1 score was slightly lower because dropout makes the learning task more challenging and discourages memorisation.

The model showed slightly better generalisation and achieved the best validation performance.

### Change optimizer to AdamW

Even after adding extra dropout, there was still some overfitting. To try to improve generalisation further, I replaced the Adam optimizer with AdamW and tested different weight decay values `1e-5, 1e-4, and 1e-3`.

In [69]:
def train_wd(model, train_data, val_data, lr, wd, epochs):

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model = model.to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)

    epoch_results = []
    best_val_f1 = -1
    best_model_state = None
    best_epoch = 0
    patience = 3
    epochs_no_improve = 0

    for epoch_num in range(epochs):

        total_loss_train = 0
        train_preds = []
        train_true_labels = []

        model.train()

        for batch in tqdm(train_data):

            batch = move_to_device(batch, device)

            word_ids = batch["word_ids"]
            char_ids = batch["char_ids"]
            labels = batch["tag_ids"]
            mask = batch["mask"].bool()

            optimizer.zero_grad()
            
            
            loss = model(word_ids, char_ids, tags=labels, mask=mask)

            loss.backward()
            optimizer.step()      

            total_loss_train += loss.item()

            with torch.no_grad():

                pred_ids = model(word_ids, char_ids, mask=mask)

                for true_labels, pred_labels, sent_mask in zip(labels.cpu().tolist(), pred_ids, mask.cpu().tolist()):

                    length = sum(sent_mask)

                    train_true_labels.append([id_to_tag[label] for label in true_labels[:length]])
                    train_preds.append([id_to_tag[label] for label in pred_labels[:length]])

        train_f1 = seq_f1_score(train_true_labels, train_preds, mode="strict", scheme=IOB2, average="macro", zero_division=0)


        model.eval()

        total_loss_val = 0
        val_true_labels = []
        val_preds = []

        with torch.no_grad():

            for batch in val_data:
            
                batch = move_to_device(batch, device)
                word_ids = batch["word_ids"]
                char_ids = batch["char_ids"]
                labels = batch["tag_ids"]
                mask = batch["mask"].bool()

                val_loss = model(word_ids, char_ids, tags=labels, mask=mask)

                pred_ids = model(word_ids, char_ids, mask=mask)

                total_loss_val += val_loss.item()

                for true_labels, pred_labels, sent_mask in zip(labels.cpu().tolist(), pred_ids, mask.cpu().tolist()):

                    length = sum(sent_mask)

                    val_true_labels.append([id_to_tag[label] for label in true_labels[:length]])

                    val_preds.append([id_to_tag[label] for label in pred_labels[:length]])

            val_f1 = seq_f1_score(val_true_labels, val_preds, mode="strict", scheme=IOB2, average="macro", zero_division=0)

            # save best model weights by validation macro F1
            if val_f1 > best_val_f1:
                best_val_f1 = val_f1
                best_epoch = epoch_num + 1
                best_model_state = copy.deepcopy(model.state_dict())
                epochs_no_improve = 0

            else:
                epochs_no_improve += 1


            results = {"Epoch": epoch_num + 1, 
                       "train_loss": round(total_loss_train / len(train_data), 3), 
                       "val_loss": round(total_loss_val / len(val_data), 3),
                       "val_f1": round(val_f1, 3),
                       "train_f1": round(train_f1, 3)}


            epoch_results.append(results)

            print(f'Epochs: {epoch_num+1} | Train Loss: {total_loss_train / len(train_data):.3f} | Train F1: {train_f1:.3f}')
            print(f'Val loss: {total_loss_val/len(val_data):.3f} | Val F1: {val_f1:.3f}')

            if epochs_no_improve >= patience:

                print(f"Early stopping after epoch {epoch_num + 1}")
                break

    # Load best weights 
    model.load_state_dict(best_model_state)

    print(f"Best model from epoch {best_epoch} Val Macro F1 = {best_val_f1:.3f}")

    return model, epoch_results, best_epoch

In [76]:
weight_decays = [1e-5, 1e-4, 1e-3]

In [77]:
wd_results = []

for wd in weight_decays:

    print(f"{wd}:")

    model_wd = CharCNNBiLSTMCRF2(word_embeddings=word_embeddings, num_char=len(char_to_id), num_tags=len(tag_to_id), 
                        char_embedding_dim=30, char_out_channel=50,
                        lstm_hidden_dim = 200, dropout=0.2).to(device)

    
    _, results_wd, best_epoch_wd = train_wd(model_wd, train_loader, valid_loader, LR,wd=wd, epochs=EPOCHS)

    best_epoch_result_wd = results_wd[best_epoch_wd - 1]

    wd_results.append({"weight_decay": wd,
                        "Best Epoch": best_epoch_wd,
                        "Train Loss": best_epoch_result_wd["train_loss"],
                        "Val Loss": best_epoch_result_wd["val_loss"],
                        "Train F1": best_epoch_result_wd["train_f1"],
                        "Validation F1": best_epoch_result_wd["val_f1"],
                        "Train-Val Gap": (best_epoch_result_wd["train_f1"] - best_epoch_result_wd["val_f1"])})  

1e-05:


100%|██████████| 439/439 [00:29<00:00, 14.88it/s]


Epochs: 1 | Train Loss: 0.271 | Train F1: 0.646
Val loss: 0.109 | Val F1: 0.794


100%|██████████| 439/439 [00:29<00:00, 14.93it/s]


Epochs: 2 | Train Loss: 0.073 | Train F1: 0.874
Val loss: 0.075 | Val F1: 0.848


...

Epochs: 5 | Train Loss: 0.023 | Train F1: 0.957
Val loss: 0.082 | Val F1: 0.859


100%|██████████| 439/439 [00:29<00:00, 15.11it/s]


Epochs: 6 | Train Loss: 0.017 | Train F1: 0.968
Val loss: 0.077 | Val F1: 0.867


100%|██████████| 439/439 [00:28<00:00, 15.19it/s]


Epochs: 7 | Train Loss: 0.012 | Train F1: 0.977
Val loss: 0.078 | Val F1: 0.875
Early stopping after epoch 7
Best model from epoch 4 Val Macro F1 = 0.876


In [78]:
wd_results_df = pd.DataFrame(wd_results).sort_values(by=["Validation F1", "Train-Val Gap"], 
                                                      ascending=[False, True])

wd_results_df

,weight_decay,Best Epoch,Train Loss,Val Loss,Train F1,Validation F1,Train-Val Gap
1,0.00010,4,0.031,0.057,0.943,0.886,0.057
0,0.00001,4,0.031,0.055,0.945,0.886,0.059
2,0.00100,4,0.031,0.072,0.941,0.876,0.065


Using `AdamW` improved the model's performance. **The best validation macro F1 score** increased from `0.880 to 0.886` when using a weight decay of 1e-5 or 1e-4.

**The train-validation gap** also decreased from `0.083 to 0.057`, suggesting that the model generalised better to unseen data and overfitted less than before.

A larger weight decay `1e-3` reduced performance, which suggests that too much regularisation prevented the model from learning useful patterns from the training data.

AdamW with a small weight decay value gave the best balance between performance and generalisation and produced the strongest results so far.

For the further experiments, I use the following hyperparameters:

- Learning rate: `1e-3`
- Dropout: `0.2`
- Optimizer: `AdamW`
- Weight decay: `1e-4`
- Early-stopping patience: `3`

These values were selected based on the previous experiments, where they provided the best balance between validation performance and generalisation. I

### Frozen and Fine-Tuned GloVe Embeddings

So far, all experiments used `fine-tuned GloVe embeddings`, meaning the embedding weights were updated during training. In this experiment, I `froze the GloVe embeddings` and kept the pre-trained vectors unchanged.

The goal was to see whether the model could benefit more from the original GloVe representations rather than adapting them to the training data.

In [84]:
class CharCNNBiLSTMCRF_frozen(nn.Module):

    def __init__(self,word_embeddings, num_char, num_tags, 
                char_embedding_dim=30, char_out_channel=50,
                lstm_hidden_dim = 200, dropout=0.2):

        super().__init__()

        self.word_emb = nn.Embedding.from_pretrained(word_embeddings, freeze=True, padding_idx=word_to_id[PAD_TOKEN])
        
        self.char_cnn = CharCNN(num_chars=num_char, char_embedding_dim=char_embedding_dim, char_out_channels=char_out_channel,
                                padding_idx=char_to_id[PAD_CHAR])

        self.dropout = nn.Dropout(dropout)

        self.bilstm = nn.LSTM(input_size=word_embeddings.shape[1] + char_out_channel, hidden_size=lstm_hidden_dim,
                              batch_first=True, bidirectional=True)

        # Convert each output into emissions for each tag
        self.emissions = nn.Linear(lstm_hidden_dim * 2, num_tags)

        self.crf = CRF(num_tags, batch_first=True)


    def forward(self, word_ids, char_ids, tags=None, mask=None):
        
        word_emb = self.word_emb(word_ids)  # [B, S, word_emb_dim]
        char_emb = self.char_cnn(char_ids)  # [B, S, char_out_channels]

        x = torch.cat([word_emb, char_emb], dim=-1)
        x = self.dropout(x)

        bilstm_out, _ = self.bilstm(x) # [B, S, hidden_dim]
        bilstm_out = self.dropout(bilstm_out)

        emissions = self.emissions(bilstm_out)

        if tags is not None:

            loss = -self.crf(emissions, tags, mask=mask.bool(), reduction="token_mean")

            return loss

        preds = self.crf.decode(emissions, mask=mask.bool())

        return preds

In [100]:
final_model_frozen = CharCNNBiLSTMCRF_frozen(word_embeddings=word_embeddings, num_char=len(char_to_id), num_tags=len(tag_to_id), 
                    char_embedding_dim=30, char_out_channel=50,
                    lstm_hidden_dim = 200, dropout=0.2).to(device)

In [101]:
trained_final_model_frozen, final_results_frozen, final_best_epoch_frozen = train_wd(final_model_frozen, train_loader, valid_loader, FINAL_LR,wd=WD, epochs=FINAL_EPOCHS)

100%|██████████| 439/439 [00:29<00:00, 14.87it/s]


Epochs: 1 | Train Loss: 0.293 | Train F1: 0.618
Val loss: 0.103 | Val F1: 0.816


100%|██████████| 439/439 [00:31<00:00, 13.90it/s]


Epochs: 2 | Train Loss: 0.091 | Train F1: 0.839
Val loss: 0.078 | Val F1: 0.852


100%|██████████| 439/439 [00:30<00:00, 14.62it/s]


...

Epochs: 6 | Train Loss: 0.034 | Train F1: 0.935
Val loss: 0.047 | Val F1: 0.898


100%|██████████| 439/439 [00:33<00:00, 13.12it/s]


Epochs: 7 | Train Loss: 0.027 | Train F1: 0.945
Val loss: 0.044 | Val F1: 0.904


100%|██████████| 439/439 [00:31<00:00, 13.89it/s]


Epochs: 8 | Train Loss: 0.023 | Train F1: 0.954
Val loss: 0.043 | Val F1: 0.904
Best model from epoch 8 Val Macro F1 = 0.904


**The validation macro F1** increased from `0.886 to 0.904`, which was the best score achieved in all experiments.

The train F1 at the best epoch was `0.954`, giving a train-validation gap of `0.050`. This is smaller than the previous gap of `0.057`, suggesting that the model was overfitting less.

A possible reason is that the pre-trained GloVe embeddings already contain useful semantic information learned from a much larger corpus. By freezing them, the model was forced to use these representations instead of changing them to fit the training set.

### BiLSTM Hidden-Dimension Experiment

The previous best model used a **BiLSTM hidden size** of `200`. In this experiment, I test smaller hidden sizes `150 and 100` to see whether reducing the model capacity could further reduce overfitting and improve generalisation.

A smaller hidden dimension means fewer trainable parameters, which can sometimes help the model focus on more general patterns instead of memorising the training data.

In [102]:
FINAL_EPOCHS = 15
FINAL_LR = 1e-3
WD = 1e-4

In [103]:
final_smodel = CharCNNBiLSTMCRF_frozen(word_embeddings=word_embeddings, num_char=len(char_to_id), num_tags=len(tag_to_id), 
                    char_embedding_dim=30, char_out_channel=50,
                    lstm_hidden_dim = 150, dropout=0.2).to(device)

In [104]:
trained_final_smodel, final_results_smodel, final_best_epoch_smodel = train_wd(final_smodel, train_loader, valid_loader, FINAL_LR,wd=WD, epochs=FINAL_EPOCHS)

100%|██████████| 439/439 [00:31<00:00, 13.74it/s]


Epochs: 1 | Train Loss: 0.311 | Train F1: 0.596
Val loss: 0.116 | Val F1: 0.789


100%|██████████| 439/439 [00:31<00:00, 13.95it/s]


Epochs: 2 | Train Loss: 0.095 | Train F1: 0.831
Val loss: 0.078 | Val F1: 0.849


100%|██████████| 439/439 [00:29<00:00, 14.71it/s]


...

Epochs: 13 | Train Loss: 0.013 | Train F1: 0.973
Val loss: 0.048 | Val F1: 0.897


100%|██████████| 439/439 [00:28<00:00, 15.64it/s]


Epochs: 14 | Train Loss: 0.011 | Train F1: 0.975
Val loss: 0.047 | Val F1: 0.899


100%|██████████| 439/439 [00:27<00:00, 15.97it/s]


Epochs: 15 | Train Loss: 0.010 | Train F1: 0.980
Val loss: 0.047 | Val F1: 0.902
Early stopping after epoch 15
Best model from epoch 12 Val Macro F1 = 0.903


In [108]:
final_smodel2 = CharCNNBiLSTMCRF_frozen(word_embeddings=word_embeddings, num_char=len(char_to_id), num_tags=len(tag_to_id), 
                    char_embedding_dim=30, char_out_channel=50,
                    lstm_hidden_dim = 100, dropout=0.2).to(device)

In [109]:
trained_final_smodel2, final_results_smodel2, final_best_epoch_smodel2 = train_wd(final_smodel2, train_loader, valid_loader, FINAL_LR, wd=WD, epochs=20)

100%|██████████| 439/439 [00:28<00:00, 15.16it/s]


Epochs: 1 | Train Loss: 0.349 | Train F1: 0.537
Val loss: 0.139 | Val F1: 0.774


100%|██████████| 439/439 [00:28<00:00, 15.40it/s]


Epochs: 2 | Train Loss: 0.110 | Train F1: 0.809
Val loss: 0.089 | Val F1: 0.831


100%|██████████| 439/439 [00:31<00:00, 14.12it/s]


...

Epochs: 6 | Train Loss: 0.044 | Train F1: 0.913
Val loss: 0.051 | Val F1: 0.888


100%|██████████| 439/439 [00:28<00:00, 15.62it/s]


Epochs: 7 | Train Loss: 0.038 | Train F1: 0.925
Val loss: 0.047 | Val F1: 0.897


100%|██████████| 439/439 [00:28<00:00, 15.54it/s]


Epochs: 8 | Train Loss: 0.034 | Train F1: 0.930
Val loss: 0.047 | Val F1: 0.893
Early stopping after epoch 8
Best model from epoch 5 Val Macro F1 = 0.897


**Reducing the hidden size did not improve validation performance**. 

**Hidden size of 150** achieved `the validation macro F1 score of 0.903`, while **hidden size of 100** achieved `0.897`. Both were lower than the previous best score of 0.904 obtained with a hidden size of 200.

The smaller models showed less overfitting, especially with a hidden size of 100, where the train-validation gap decreased to 0.022. However, this came at the cost of lower validation performance.

This suggests that reducing the model size too much limits its ability to learn useful contextual information. Although the larger model showed a slightly bigger gap, it achieved the best validation F1 score and therefore remained the preferred configuration.

### Evaluation Across Random Seeds

To make sure the results were not caused by a lucky random seed, I trained the final model with different seeds.

In [ ]:
seeds = [123, 2026]
seeds_scores = []

for seed in seeds:
    
    set_seed(seed)

    model = CharCNNBiLSTMCRF_frozen(word_embeddings=word_embeddings, num_char=len(char_to_id), num_tags=len(tag_to_id),
                                    char_embedding_dim=30, char_out_channel=50,
                                    lstm_hidden_dim=200, dropout=0.2).to(device)

    _, results_seeds, best_epoch_seeds = train_wd(model, train_loader, valid_loader, FINAL_LR, wd=WD, epochs=15)

    best_result_seed = results_seeds[best_epoch_seeds - 1]

    seeds_scores.append({"seed": seed,
                        "Best Epoch": best_epoch_seeds,
                        "Train Loss": best_result_seed["train_loss"],
                        "Val Loss": best_result_seed["val_loss"],
                        "Train F1": best_result_seed["train_f1"],
                        "Validation F1": best_result_seed["val_f1"],
                        "Train-Val Gap": (best_result_seed["train_f1"] - best_result_seed["val_f1"])})  

100%|██████████| 439/439 [00:29<00:00, 14.80it/s]


Epochs: 1 | Train Loss: 0.302 | Train F1: 0.605
Val loss: 0.117 | Val F1: 0.777


100%|██████████| 439/439 [00:30<00:00, 14.55it/s]


Epochs: 2 | Train Loss: 0.094 | Train F1: 0.831
Val loss: 0.073 | Val F1: 0.856


100%|██████████| 439/439 [00:28<00:00, 15.29it/s]


...

Epochs: 10 | Train Loss: 0.020 | Train F1: 0.958
Val loss: 0.044 | Val F1: 0.906


100%|██████████| 439/439 [00:27<00:00, 16.22it/s]


Epochs: 11 | Train Loss: 0.016 | Train F1: 0.965
Val loss: 0.047 | Val F1: 0.906


100%|██████████| 439/439 [00:27<00:00, 16.01it/s]


Epochs: 12 | Train Loss: 0.014 | Train F1: 0.971
Val loss: 0.047 | Val F1: 0.906
Early stopping after epoch 12
Best model from epoch 9 Val Macro F1 = 0.907


In [113]:
seeds_results_df = pd.DataFrame(seeds_scores).sort_values(by=["Validation F1", "Train-Val Gap"], 
                                                      ascending=[False, True])

seeds_results_df

,seed,Best Epoch,Train Loss,Val Loss,Train F1,Validation F1,Train-Val Gap
1,2026,9,0.023,0.044,0.952,0.907,0.045
0,123,12,0.014,0.049,0.970,0.907,0.063


The validation F1 scores were very close across runs, showing that the model performs consistently. The best run achieved a validation macro F1 score of `0.907`, which is slightly better than the previous best result of 0.904.

Overall, the small variation between runs suggests that the model is stable and that the improvements seen in previous experiments are reliable.

### Final Model Selection

The final model was chosen based mainly on validation macro F1 score. When two models achieved the same validation F1, I also considered the validation loss and the train-validation gap.

The final model used the following settings:

* Random seed: `2026`
* Frozen GloVe embeddings
* Character embedding dimension: `30`
* Character CNN output channels: `50`
* BiLSTM hidden dimension: `200`
* Dropout: `0.2`
* Optimizer: `AdamW`
* Learning rate: `1e-3`
* Weight decay: `1e-4`
* Early stopping patience: `3`

Both seed `123` and seed `2026` achieved the highest validation macro F1 score of `0.907`. However, seed `2026` achieved a slightly lower validation loss `0.044` and a smaller train-validation gap `0.045`. For this reason, the model from seed `2026` at epoch `9` was selected as the final model.

This model will now be evaluated on the test set, which has not been used during training or model selection, to obtain the final performance estimate.


In [ ]:
final_model = CharCNNBiLSTMCRF_frozen(word_embeddings=word_embeddings, num_char=len(char_to_id), num_tags=len(tag_to_id),
                                    char_embedding_dim=30, char_out_channel=50,
                                    lstm_hidden_dim=200, dropout=0.2).to(device)

In [115]:
set_seed(2026)

In [116]:
final_model, final_results, final_best_epoch = train_wd(final_model, train_loader, valid_loader, FINAL_LR, wd=WD, epochs=15)

100%|██████████| 439/439 [00:30<00:00, 14.27it/s]


Epochs: 1 | Train Loss: 0.315 | Train F1: 0.586
Val loss: 0.113 | Val F1: 0.811


100%|██████████| 439/439 [00:27<00:00, 16.16it/s]


Epochs: 2 | Train Loss: 0.097 | Train F1: 0.825
Val loss: 0.078 | Val F1: 0.851


100%|██████████| 439/439 [00:27<00:00, 15.72it/s]


...

Epochs: 13 | Train Loss: 0.013 | Train F1: 0.972
Val loss: 0.046 | Val F1: 0.904


100%|██████████| 439/439 [00:28<00:00, 15.50it/s]


Epochs: 14 | Train Loss: 0.012 | Train F1: 0.973
Val loss: 0.047 | Val F1: 0.908


100%|██████████| 439/439 [00:29<00:00, 14.88it/s]


Epochs: 15 | Train Loss: 0.010 | Train F1: 0.978
Val loss: 0.052 | Val F1: 0.902
Early stopping after epoch 15
Best model from epoch 12 Val Macro F1 = 0.909


In [118]:
torch.save(final_model.state_dict(), "../models/bilstm_weights.pt")

In [121]:
final_results

[{'Epoch': 1,
  'train_loss': 0.315,
  'val_loss': 0.113,
  'val_f1': np.float64(0.811),
  'train_f1': np.float64(0.586)},
 {'Epoch': 2,
  'train_loss': 0.097,
  'val_loss': 0.078,
  'val_f1': np.float64(0.851),
  'train_f1': np.float64(0.825)},
 {'Epoch': 3,
  'train_loss': 0.07,
  'val_loss': 0.062,
  'val_f1': np.float64(0.875),
  'train_f1': np.float64(0.872)},
 {'Epoch': 4,
  'train_loss': 0.055,
  'val_loss': 0.053,
  'val_f1': np.float64(0.886),
  'train_f1': np.float64(0.897)},
 {'Epoch': 5,
  'train_loss': 0.045,
  'val_loss': 0.048,
  'val_f1': np.float64(0.897),
  'train_f1': np.float64(0.915)},
 {'Epoch': 6,
  'train_loss': 0.038,
  'val_loss': 0.047,
  'val_f1': np.float64(0.899),
  'train_f1': np.float64(0.926)},
 {'Epoch': 7,
  'train_loss': 0.032,
  'val_loss': 0.045,
  'val_f1': np.float64(0.9),
  'train_f1': np.float64(0.936)},
 {'Epoch': 8,
  'train_loss': 0.028,
  'val_loss': 0.044,
  'val_f1': np.float64(0.906),
  'train_f1': np.float64(0.946)},
 {'Epoch': 9,
  'tr

In [124]:
final_results_df = pd.DataFrame(final_results).sort_values(by=["val_f1"], 
                                                      ascending=False)

final_results_df

,Epoch,train_loss,val_loss,val_f1,train_f1
11,12,0.015,0.047,0.909,0.969
13,14,0.012,0.047,0.908,0.973
9,10,0.020,0.046,0.907,0.959
7,8,0.028,0.044,0.906,0.946
10,11,0.017,0.048,0.905,0.964
8,9,0.023,0.042,0.905,0.955
12,13,0.013,0.046,0.904,0.972
14,15,0.010,0.052,0.902,0.978
6,7,0.032,0.045,0.900,0.936
5,6,0.038,0.047,0.899,0.926


In [125]:
final_results_df.to_csv("../results/bilstm_final_results", index=False)

The final model achieved its best validation macro F1 score of `0.909 at epoch 12`. Training continued until epoch 15, where early stopping stopped the training because the validation score had not improved for three consecutive epochs. 

Looking at the results, the training F1 continued to increase after epoch 12, but the validation F1 started to drop slightly. This suggests that the model was starting to fit the training data too closely rather than learning patterns that generalise well to unseen data.

**The train-validation gap** was around `0.06`. While the model performed slightly better on the training data, it still achieved the highest validation F1 score, suggesting good overall generalisation.